<a href="https://colab.research.google.com/github/Noisy77-pixel/urdu-ocr-codesaviours-si26-bilal/blob/main/SI26_Week3_Bilal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade torch torchvision torchaudio sentencepiece transformers pillow pandas
!pip install torch==2.2.2+cu121 torchvision==0.17.2+cu121 transformers sentencepiece pillow pandas --extra-index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.2.2%2Bcu121-cp312-cp312-linux_x86_64.whl (757.2 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.17.2%2Bcu121-cp312-cp312-linux_x86_64.whl (7.0 MB)
  Attempting uninstall: torch
    Found existing installation: torch 2.13.0
    Uninstalling torch-2.13.0:
      Successfully uninstalled torch-2.13.0
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.28.0
    Uninstalling torchvision-0.28.0:
      Successfully uninstalled torchvision-0.28.0


In [2]:
import torch
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import ViTImageProcessor, RobertaTokenizer, TrOCRProcessor
import os
import sys

print(f"Verifying PyTorch installation: {torch.__version__}")

class UrduOCRDataset(Dataset):
    """
    PyTorch Dataset for Urdu OCR using TrOCR processor.
    Handles dynamic file path mapping to find images across subdirectories.
    """
    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        self.path_mapping = {}

        print("Mapping image paths across all subdirectories...")
        for root, _, files in os.walk('data'):
            for f in files:
                if f.endswith(('.png', '.jpg', '.jpeg')):
                    self.path_mapping[f] = os.path.join(root, f)

        # Filter out missing files based on what was actually found in the folder
        self.data['filename'] = self.data['image'].apply(os.path.basename)
        initial_count = len(self.data)
        self.data = self.data[self.data['filename'].isin(self.path_mapping.keys())].reset_index(drop=True)

        if len(self.data) < initial_count:
            print(f"⚠️ Filtered out {initial_count - len(self.data)} missing images.")
        print(f'✅ Final Dataset size: {len(self.data)} samples.')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        filename = row['filename']
        image_path = self.path_mapping[filename]

        image = Image.open(image_path).convert('RGB')

        # Use return_tensors='pt' explicitly, fallback to np if torch link fails
        try:
            encoding = self.processor(image, return_tensors='pt')
            pixel_values = encoding.pixel_values.squeeze()
        except Exception:
            encoding = self.processor(image, return_tensors='np')
            pixel_values = torch.from_numpy(encoding.pixel_values).squeeze()

        # Tokenize the text label
        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128,
            truncation=True
        ).input_ids

        return {'pixel_values': pixel_values, 'labels': torch.tensor(labels)}

print('UrduOCRDataset class defined successfully!')


[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2+cu121
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[transformers] `ViTImageProcessor` requires torchvision (not installed); falling back to `ViTImageProcessorPil` for backward compatibility. Install torchvision to use the default backend, or import `ViTImageProcessorPil` directly to silence this warning.


Verifying PyTorch installation: 2.2.2+cu121
UrduOCRDataset class defined successfully!


In [3]:
try:
    # Use slow tokenizer explicitly to avoid sentencepiece backend errors in Colab
    image_processor = ViTImageProcessor.from_pretrained('microsoft/trocr-base-printed')
    tokenizer = RobertaTokenizer.from_pretrained('microsoft/trocr-base-printed', use_fast=False)
    processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

    dataset = UrduOCRDataset('data/labels.csv', processor)

    if len(dataset) > 0:
        sample = dataset[0]
        print(f'\n🚀 Success! Sample loaded successfully.')
        print(f'Pixel values shape: {sample["pixel_values"].shape}')
        print(f'Labels length: {len(sample["labels"])}')
    else:
        print("❌ Dataset is empty. Make sure your 'data' folder and 'labels.csv' are present.")
except Exception as e:
    print(f'❌ Error during initialization or test: {e}')


Mapping image paths across all subdirectories...
⚠️ Filtered out 20 missing images.
✅ Final Dataset size: 403 samples.

🚀 Success! Sample loaded successfully.
Pixel values shape: torch.Size([3, 384, 384])
Labels length: 128


In [4]:
if 'dataset' in locals() and len(dataset) > 0:
    # 80% for training, 20% for testing
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size

    train_dataset, test_dataset = torch.utils.data.random_split(
        dataset, [train_size, test_size]
    )

    print(f'\nTotal samples:    {len(dataset)}')
    print(f'Training samples: {train_size}')
    print(f'Testing samples:  {test_size}')

    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=4)

    print(f'Training batches:  {len(train_loader)}')
    print(f'Testing batches:   {len(test_loader)}')

    # Quick sanity check — load one batch
    batch = next(iter(train_loader))
    print(f'\nBatch pixel_values shape: {batch["pixel_values"].shape}')
    print(f'Batch labels shape:       {batch["labels"].shape}')
    print('\n✅ DataLoaders are working! Week 3 is COMPLETE!')
else:
    print("❌ Cannot create DataLoaders because 'dataset' is missing or empty. Did you run Cell 3?")


Total samples:    403
Training samples: 322
Testing samples:  81
Training batches:  81
Testing batches:   21

Batch pixel_values shape: torch.Size([4, 3, 384, 384])
Batch labels shape:       torch.Size([4, 128])

✅ DataLoaders are working! Week 3 is COMPLETE!
